In [1]:
import numpy as np
import re
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences



In [2]:
with open("1661-0.txt",'r',encoding='utf-8') as f: 
    text=f.read()

In [4]:
import numpy as np
import re
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# ---------------------------
# 1. Clean text
# ---------------------------
text = text.lower()
text = re.sub(r'[^a-z.!? ]', '', text)

# ---------------------------
# 2. Split into sentences
# ---------------------------
sentences = re.split(r'[.!?]', text)
sentences = [s.strip() for s in sentences if s.strip()]

print("Total sentences:", len(sentences))

# ---------------------------
# 3. Tokenizer
# ---------------------------
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)

# ---------------------------
# 4. Generate sequences (SAFE)
# ---------------------------
MAX_LEN = 50   # 🔥 critical to avoid memory explosion

input_sequences = []

for line in sentences:
    token_list = tokenizer.texts_to_sequences([line])[0]
    
    for i in range(1, len(token_list)):
        seq = token_list[max(0, i - MAX_LEN): i + 1]
        input_sequences.append(seq)

print("Total sequences:", len(input_sequences))

# ---------------------------
# 5. Pad sequences
# ---------------------------
input_sequences = pad_sequences(
    input_sequences,
    maxlen=MAX_LEN,
    padding='pre'
)

print("Shape:", input_sequences.shape)

Total sentences: 1
Total sequences: 109020
Shape: (109020, 50)


In [9]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_LEN = 100  # or 200, 300 depending on your problem

input_sequences = pad_sequences(
    input_sequences,
    maxlen=MAX_LEN,
    padding='pre',
    truncating='pre'
)

In [5]:
X = input_sequences[:, :-1]
y = input_sequences[:, -1]


In [6]:
from tensorflow.keras.utils import to_categorical

total_words = len(tokenizer.word_index) + 1
y = to_categorical(y, num_classes=total_words)

In [11]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

model = Sequential()

model.add(Embedding(total_words, 50, input_length=MAX_LEN-1))
model.add(LSTM(100))
model.add(Dense(total_words, activation='softmax'))

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

d:\Saran\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [12]:
model.fit(X, y, epochs=10, verbose=1)

Epoch 1/10
3407/3407 ━━━━━━━━━━━━━━━━━━━━ 275s 79ms/step - accuracy: 0.0697 - loss: 6.3416
Epoch 2/10
3407/3407 ━━━━━━━━━━━━━━━━━━━━ 306s 90ms/step - accuracy: 0.1123 - loss: 5.6558
Epoch 3/10
3407/3407 ━━━━━━━━━━━━━━━━━━━━ 290s 85ms/step - accuracy: 0.1352 - loss: 5.2895
Epoch 4/10
3407/3407 ━━━━━━━━━━━━━━━━━━━━ 230s 67ms/step - accuracy: 0.1503 - loss: 5.0071
Epoch 5/10
3407/3407 ━━━━━━━━━━━━━━━━━━━━ 242s 71ms/step - accuracy: 0.1649 - loss: 4.7595
Epoch 6/10
3407/3407 ━━━━━━━━━━━━━━━━━━━━ 242s 71ms/step - accuracy: 0.1777 - loss: 4.5352
Epoch 7/10
3407/3407 ━━━━━━━━━━━━━━━━━━━━ 292s 86ms/step - accuracy: 0.1913 - loss: 4.3284
Epoch 8/10
3407/3407 ━━━━━━━━━━━━━━━━━━━━ 274s 80ms/step - accuracy: 0.2072 - loss: 4.1358
Epoch 9/10
3407/3407 ━━━━━━━━━━━━━━━━━━━━ 243s 71ms/step - accuracy: 0.2255 - loss: 3.9564
Epoch 10/10
3407/3407 ━━━━━━━━━━━━━━━━━━━━ 241s 71ms/step - accuracy: 0.2444 - loss: 3.7873


In [13]:
def predict_next_word(text):

    token_list = tokenizer.texts_to_sequences([text])[0]

    token_list = pad_sequences(
        [token_list],
        maxlen=MAX_LEN-1,
        padding='pre'
    )

    pred = model.predict(token_list)

    predicted_index = np.argmax(pred)

    for word, index in tokenizer.word_index.items():
        if index == predicted_index:
            return word

In [18]:
predict_next_word(" who loathed every form of society with his")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step


'work'

In [16]:
seed = "were sufficient to absorb all my"

next_word = predict_next_word(seed)

print("Next word:", next_word)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
Next word: hair
